# FleetLogix · Avance #3 · Data Warehouse (Snowflake)

**Notebook de entrega** — modelo estrella, ETL, calidad, SCD tipo 2, ingresos simulados y auditoría.

> Los resultados numéricos expuestos son los validados y reportados por el estudiante durante el desarrollo. Las celdas SQL/Python siguientes son **opcionales** para reproducir lecturas sobre la cuenta Snowflake configurada.


## 1. Alcance y arquitectura

**Origen:** PostgreSQL (`fleetlogix_db`): seis tablas operacionales. **Destino:** Snowflake `FLEETLOGIX_DWH` (`ANALYTICS`, `STAGING`, `CONTROL`), warehouse X-Small `FLEETLOGIX_WH`.

**Grano del hecho:** una entrega completada, `DELIVERY_ID`. Dimensiones: `DIM_DATE`, `DIM_TIME`, `DIM_VEHICLE`, `DIM_DRIVER` (SCD2), `DIM_ROUTE`, `DIM_CUSTOMER`.

**Reglas:** cliente = hash de nombre+dirección normalizados (identidad provisional); combustible y kilómetros distribuidos por peso entre entregas de cada viaje, con conciliación; ingreso sintético COP por entrega = `ROUND(7000 + 20*peso_kg + 300*km_asignados, 2)`.


## 2. Evidencias de la carga

| Métrica | Valor confirmado |
|---|---:|
| Calendario / horas | 2.557 / 24 |
| Vehículos / conductores / rutas | 200 / 400 / 50 |
| Identidades cliente | 389.334 |
| Entregas / viajes | 400.000 / 100.000 |
| Distancia asignada | 4.701.486,980000 km |
| Combustible asignado | 633.064,820000 litros |
| Ingresos **sintéticos** | 6.270.850.032,24 COP |
| MERGE idéntico (hechos) | 0 inserciones / 0 actualizaciones |
| Integridad dimensional | 0 referencias nulas |
| Orquestador `--full` | 9 pasos SUCCESS / 0 FAILED |
| Delta de diferencias | 0 nuevas / 0 modificadas / 400.000 iguales |


## 3. Conexión de solo lectura (opcional)

La cuenta se autenticó con Google mediante OAuth. Las variables Snowflake permanecen en `.env` y **no** se publican en el repositorio. Ejecutar desde la raíz del proyecto con las dependencias instaladas si se desea regenerar los resultados.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / '.env')

# Ejecutar solo si se desea consultar Snowflake desde este notebook.
# Requiere snowflake-connector-python y acceso OAuth en navegador.
def conectar_snowflake():
    import snowflake.connector
    return snowflake.connector.connect(
        account=os.environ['SNOWFLAKE_ACCOUNT'],
        user=os.environ['SNOWFLAKE_USER'],
        authenticator='OAUTH_AUTHORIZATION_CODE',
        role=os.getenv('SNOWFLAKE_ROLE', 'SYSADMIN'),
        warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
        database='FLEETLOGIX_DWH',
        schema='ANALYTICS',
        enable_server_session_keep_alive_auto_detection=False,
    )


In [ ]:
# Lectura opcional: descomentar para ejecutar.
# with conectar_snowflake() as conn:
#     with conn.cursor() as cur:
#         cur.execute('''
#             SELECT COUNT(*) AS entregas,
#                    COUNT(DISTINCT delivery_id) AS ids_unicos,
#                    ROUND(SUM(allocated_distance_km),6) AS km,
#                    ROUND(SUM(allocated_fuel_liters),6) AS litros,
#                    ROUND(SUM(estimated_revenue_cop),2) AS ingresos_cop
#             FROM ANALYTICS.FACT_DELIVERIES
#         ''')
#         print(cur.fetchone())


## 4. SCD tipo 2 y pruebas de idempotencia

La prueba aislada del cargador Python generó dos versiones para el `DRIVER_ID=1`: versión histórica y versión vigente. Segunda pasada: 401 versiones totales, 400 actuales, 1 histórica, cero anomalías temporales. Se comprobó que las tablas reales continuaron con 400 conductores y 400.000 hechos.

**Importante:** la fecha inicial 1900 del primer snapshot es un ancla técnica, no evidencia de datos reales de ese año.


## 5. Analytics, auditoría y protección de acceso

`16_finalize_analytics.sql` construye `AGG_DRIVER_HOURLY` y `AGG_MONTHLY_ROUTE`, más las vistas seguras `VW_OPERATIONS_DELIVERIES` y `VW_SALES_MONTHLY_REVENUE`. OPERATIONS incluye dirección de entrega para personal autorizado; SALES solo ve indicadores agregados de ingresos **simulados**, sin nombres ni direcciones.

`17_roles_opcionales.sql` define roles separados con concesión de lectura exclusiva sobre sus vistas; la asignación efectiva de usuarios exige intervención administrativa.

El orquestador `10_run_etl_pipeline.py --full` finalizó 9/9 pasos satisfactoriamente, con registro en `CONTROL.ETL_RUNS` y `CONTROL.ETL_RUN_STEPS`.


## 6. Limitaciones y próximos pasos

- Incrementalidad: comparación por hashes y delta vacío confirmados; escritura de delta no vacío aún sin ensayo real. Sin CDC en origen (`updated_at` no disponible de forma uniforme).
- Automatización nocturna desatendida: no habilitada; cargadores todavía con OAuth interactivo. Requiere usuario de servicio, credenciales no interactivas y scheduler seguro.
- Time Travel: retención configurada a 1 día; requerimiento de 30 días condicionado a Snowflake Enterprise o superior.
- Agregados: snapshots, deben regenerarse tras cambios de hechos.
- Ingresos COP son **estimaciones académicas**, no registros contables.

**Conclusión:** el modelo dimensional, la carga inicial, las conciliaciones, la orquestación manual auditada, los controles SCD2 y la idempotencia están demostrados. Las limitaciones indicadas no deben ocultarse en la presentación.
